# AusGrid PV Monthly Loader & Concatenator

Loads the 12 monthly AusGrid distributed-PV telemetry zips (`jm_unswnmis<mon><yy>.zip`,
Aug 2024 – Jul 2025), each containing a single same-stem CSV, stacks them into one
DataFrame with a `month` column derived from the filename, harmonises dtypes, and
(optionally) writes a single Parquet for fast re-loading.

**Observed schema** (from the example file. long / per-phase format):

| column | meaning | notes |
|---|---|---|
| `serial` | inverter/meter serial. **the site identifier** | int64 |
| `MeasureTime` | timestamp, 5-min cadence, ISO-8601 with `Z` | [!] `Z`=UTC but values look like **local** time (see DQ notebook §5) |
| `Vphase` | phase label `A`/`B`/`C` | 3 rows per timestamp |
| `Volts` | phase voltage (V) | |
| `Curr` | phase current (A) | |
| `ReactPow` | reactive power (VAr), per phase | all-negative & coarsely quantised in the sample |
| `ActivePow` | active power (W), per phase | ≥0; quantised to ~5.105 W steps |


## Config

In [ ]:
from pathlib import Path

# Folder containing the monthly zips
DATA_DIR = Path(r"C:\Users\z3553082\OneDrive - UNSW\Documents/CICCADA - Data\Ausgrid")

# Filename glob for the monthly zips
ZIP_GLOB = "jm_unswnmis*.zip"

# Where to write the combined Parquet (set to None to skip writing).
PARQUET_OUT = DATA_DIR / "ausgrid_pv_combined.parquet"

# Extract zips to a temp dir (True, recommended) or alongside the zips (False).
USE_TEMP_DIR = True

# If True, keep MeasureTime as tz-naive (recommended until the UTC-vs-local
# question is resolved with AusGrid — see DQ notebook §5).
KEEP_TIME_NAIVE = True

## Imports

In [ ]:
import re
import zipfile
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


## Filename to month parser
Maps `jm_unswnmisapr25` to period `2025-04`. Robust to upper/lower case and to the
`.csv`/`.zip` suffix.

In [ ]:
_MONTHS = {m: i for i, m in enumerate(
    ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"], start=1)}

_STEM_RE = re.compile(r"jm_unswnmis([a-z]{3})(\d{2})", re.IGNORECASE)

def month_from_name(name: str) -> pd.Period:
    """Extract a monthly Period from a jm_unswnmis<mon><yy> file/zip name."""
    m = _STEM_RE.search(Path(name).stem)
    if not m:
        raise ValueError(f"Could not parse month from: {name!r}")
    mon, yy = m.group(1).lower(), int(m.group(2))
    if mon not in _MONTHS:
        raise ValueError(f"Unknown month token {mon!r} in {name!r}")
    year = 2000 + yy
    return pd.Period(year=year, month=_MONTHS[mon], freq="M")

# quick test
for t in ["jm_unswnmisapr25.zip", "jm_unswnmisAUG24.csv", "jm_unswnmisjan25"]:
    print(f"{t:28s} -> {month_from_name(t)}")


## Canonical dtypes & one-file reader
Reading every month with an explicit dtype map prevents pandas from inferring
`serial` as float (it will, if any month has a stray null) or mixing object/float
across months. `MeasureTime` is parsed once, here.

In [ ]:
# Expected columns and the dtype we force them to. 
# Extra/missing columns are reported rather than silently dropped, so schema drift across months is visible.
EXPECTED_DTYPES = {
    "serial":    "int64",
    "Vphase":    "category",
    "Volts":     "float64",
    "Curr":      "float64",
    "ReactPow":  "float64",
    "ActivePow": "float64",
}
TIME_COL = "MeasureTime"

def read_one_csv(csv_path: Path) -> pd.DataFrame:
    """Read a single monthly CSV with stable dtypes and a parsed timestamp."""
    df = pd.read_csv(csv_path)

    # --- schema drift report (non-fatal) ---
    cols = set(df.columns)
    expected = set(EXPECTED_DTYPES) | {TIME_COL}
    missing, extra = expected - cols, cols - expected
    if missing:
        print(f"  [warn] {csv_path.name}: MISSING columns {sorted(missing)}")
    if extra:
        print(f"  [warn] {csv_path.name}: EXTRA columns {sorted(extra)} (kept as-is)")

    # --- timestamp ---
    if TIME_COL in df.columns:
        # utc=True correctly parses the trailing 'Z'; we then drop tz to stay
        # naive while the UTC-vs-local question is open (see config note).
        ts = pd.to_datetime(df[TIME_COL], utc=True, errors="coerce")
        n_bad = ts.isna().sum()
        if n_bad:
            print(f"  [warn] {csv_path.name}: {n_bad} unparseable timestamps -> NaT")
        df[TIME_COL] = ts.dt.tz_localize(None) if KEEP_TIME_NAIVE else ts

    # --- numeric / categorical dtypes ---
    for col, dt in EXPECTED_DTYPES.items():
        if col not in df.columns:
            continue
        if dt == "int64":
            # coerce via float first to tolerate stray non-numeric, then check
            s = pd.to_numeric(df[col], errors="coerce")
            if s.isna().any():
                print(f"  [warn] {csv_path.name}: {col} has non-integer/null values; kept as float64")
                df[col] = s.astype("float64")
            else:
                df[col] = s.astype("int64")
        elif dt == "category":
            df[col] = df[col].astype("category")
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(dt)

    return df


## Extract + read every month, then concatenate

In [ ]:
def iter_csv_from_zip(zip_path: Path, workdir: Path) -> Path:
    """Extract the single CSV inside a monthly zip and return its path."""
    with zipfile.ZipFile(zip_path) as zf:
        members = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not members:
            raise FileNotFoundError(f"No CSV inside {zip_path.name}")
        if len(members) > 1:
            print(f"  [warn] {zip_path.name}: {len(members)} CSVs inside; using {members[0]}")
        target = members[0]
        zf.extract(target, path=workdir)
        return workdir / target


def load_all(data_dir: Path, zip_glob: str) -> pd.DataFrame:
    zips = sorted(data_dir.glob(zip_glob))
    if not zips:
        raise FileNotFoundError(f"No zips matching {zip_glob!r} in {data_dir}")
    print(f"Found {len(zips)} monthly zips.\n")

    frames = []
    ctx = tempfile.TemporaryDirectory() if USE_TEMP_DIR else None
    workdir = Path(ctx.name) if ctx else data_dir
    try:
        for zp in zips:
            period = month_from_name(zp.name)
            print(f"- {zp.name}  (month={period})")
            csv_path = iter_csv_from_zip(zp, workdir)
            df = read_one_csv(csv_path)
            df["month"] = period                        # PeriodM from filename
            df["source_file"] = zp.stem                 # provenance
            frames.append(df)
            if USE_TEMP_DIR:
                try: csv_path.unlink()
                except OSError: pass
    finally:
        if ctx:
            ctx.cleanup()

    combined = pd.concat(frames, ignore_index=True)
    # store month as a clean categorical string for Parquet friendliness
    combined["month"] = combined["month"].astype(str).astype("category")
    return combined


combined = load_all(DATA_DIR, ZIP_GLOB)
print(f"\nCombined shape: {combined.shape[0]:,} rows x {combined.shape[1]} cols")
combined.head()


## Sanity summary

In [ ]:
print("dtypes:")
print(combined.dtypes, "\n")

print("rows per month:")
print(combined["month"].value_counts().sort_index(), "\n")

if "serial" in combined:
    print(f"distinct sites (serial): {combined['serial'].nunique():,}")
if TIME_COL in combined:
    print(f"time span: {combined[TIME_COL].min()}  ->  {combined[TIME_COL].max()}")
if "Vphase" in combined:
    print("phase counts:", combined["Vphase"].value_counts().to_dict())

print("\nglobal null rates (%):")
print((combined.isna().mean() * 100).round(3))


## Optional: write a single Parquet for fast re-loading

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(combined, preserve_index=False)
pq.write_table(table, str(PARQUET_OUT), compression="snappy")
size_mb = PARQUET_OUT.stat().st_size / 1e6
print(f"Wrote {PARQUET_OUT}  ({size_mb:,.1f} MB)")

chk = pq.read_table(str(PARQUET_OUT)).to_pandas()
assert chk.shape == combined.shape, "Parquet round-trip shape mismatch!"
print("Round-trip OK:", chk.shape)

In [ ]:
'''
if PARQUET_OUT is not None:
    PARQUET_OUT.parent.mkdir(parents=True, exist_ok=True)
    combined.to_parquet(PARQUET_OUT, engine="pyarrow", index=False)
    size_mb = PARQUET_OUT.stat().st_size / 1e6
    print(f"Wrote {PARQUET_OUT}  ({size_mb:,.1f} MB)")
    # round-trip check
    chk = pd.read_parquet(PARQUET_OUT, engine="pyarrow")
    assert chk.shape == combined.shape, "Parquet round-trip shape mismatch!"
    print("Round-trip OK:", chk.shape)
else:
    print("PARQUET_OUT is None — skipped writing.")
'''

In [ ]:
import pandas as pd, pyarrow as pa
print("pandas:", pd.__version__)
print("pyarrow:", pa.__version__)